# RSNA 2025 — Pretrained Pipeline Evaluation on 100 Patients

**Input**: `D:\Training_Data` — 100 DICOM folders (one per patient, named by SeriesInstanceUID)  
**Labels**: `train.csv` — ground-truth binary labels (13 locations + Aneurysm Present)  
**Output**: Accuracy, Precision, Recall, F1, AUC-ROC + two confusion matrices

### Pipeline (exact winners pipeline — unchanged from Pretrained.ipynb)
1. **DICOM → NIfTI** via `dcm2niix`
2. **Vessel segmentation** via `VesselSegmentationPredictor`
3. **ROI crop** from segmentation, z-score normalised
4. **ROI classification** via `AneurysmVesselSegROILitModuleTransformer` (fold ensemble)
5. **Output**: `polars.DataFrame` with 14 columns — 13 location probs + 1 presence prob

### Memory strategy
- Each volume is passed through `predict()` once, only the 14 probabilities are stored, then the volume is discarded.
- `torch.cuda.empty_cache()` + `gc.collect()` called after every patient.

### Evaluation index
- `PATIENT_START = 0`, `PATIENT_END = 100` — change these to test a subset.

## Cell 1 — Configure paths (same as Pretrained.ipynb + new batch paths)

In [ ]:
import os
from pathlib import Path

# ============================================================
# ORIGINAL PATHS — identical to Pretrained.ipynb
# ============================================================
REPO_ROOT = Path(r"C:\Users\maila\Desktop\RSNA_Major\rsna2025_main")
DEVICE    = "cuda:0"

VESSEL_SPARSE_MODEL_DIR  = str(REPO_ROOT / "nnunet-vessel-grouping-da7")
VESSEL_PRIMARY_MODEL_DIR = str(REPO_ROOT / "nnunet-da3-sklr-ep800")
VESSEL_ADDL_MODEL_DIRS   = str(REPO_ROOT / "nnunet-da6-sklr-w3-tv07")

ROI_EXPERIMENT = "251013-seg_tf-v4-nnunet_truncate1_preV6_1-ex_dav6w3-m32g64-e25-w01_005_1-s128_256_256"
ROI_FOLDS      = "0,1,2,3"
ROI_TTA        = "2"
CONFIG_DIR     = str(REPO_ROOT / "configs")
RSNA_DEBUG     = "1"

# ============================================================
# NEW — Batch evaluation paths
# ============================================================
TRAINING_DATA_ROOT = Path(r"D:\Training_Data")     
TRAIN_CSV_PATH     = REPO_ROOT / "train.csv"       

# Patient index slice — change to test a subset, e.g. [0:10]
PATIENT_START = 0
PATIENT_END   = 100   

print(f"REPO_ROOT          : {REPO_ROOT}")
print(f"TRAINING_DATA_ROOT : {TRAINING_DATA_ROOT}")
print(f"TRAIN_CSV_PATH     : {TRAIN_CSV_PATH}")
print(f"DEVICE             : {DEVICE}")
print(f"Sparse model dir   : {VESSEL_SPARSE_MODEL_DIR}")
print(f"Primary model dir  : {VESSEL_PRIMARY_MODEL_DIR}")
print(f"Additional model   : {VESSEL_ADDL_MODEL_DIRS}")
print(f"ROI experiment     : {ROI_EXPERIMENT}")
print(f"ROI folds          : {ROI_FOLDS}")
print(f"ROI TTA            : {ROI_TTA}")
print(f"Config dir         : {CONFIG_DIR}")
print(f"Patients to test   : [{PATIENT_START}:{PATIENT_END}]")

## Cell 2 — Bootstrap sys.path and PROJECT_ROOT (identical to Pretrained.ipynb)

In [3]:
import sys

# Ensure repo root and the bundled nnUNet fork are importable
_paths_to_add = [
    str(REPO_ROOT),
    str(REPO_ROOT / "nnUNet"),
]
for _p in reversed(_paths_to_add):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Set PROJECT_ROOT so Hydra / configs can resolve ${oc.env:PROJECT_ROOT}
os.environ["PROJECT_ROOT"] = str(REPO_ROOT)

# Change working directory to repo root (rootutils / Hydra expect this)
os.chdir(REPO_ROOT)

print("sys.path[0]   :", sys.path[0])
print("sys.path[1]   :", sys.path[1])
print("PROJECT_ROOT  :", os.environ["PROJECT_ROOT"])
print("cwd           :", os.getcwd())

## Cell 3 — Set environment variables

In [4]:
import logging
from tqdm import tqdm
import gc
logging.getLogger('pytorch_lightning').setLevel(logging.WARNING)
logging.getLogger('nnunet').setLevel(logging.WARNING)
# ── nnUNet paths ───────────────────────────────────────────────────────────
os.environ["nnUNet_raw"]           = str(REPO_ROOT / "logs" / "nnUNet_raw")
os.environ["nnUNet_preprocessed"]  = str(REPO_ROOT / "logs" / "nnUNet_preprocessed")
os.environ["nnUNet_results"]       = str(REPO_ROOT / "logs" / "nnUNet_results")

# ── Vessel segmentation ────────────────────────────────────────────────────
os.environ["VESSEL_DEVICE"]                      = DEVICE
os.environ["VESSEL_DEVICES"]                     = DEVICE.replace("cuda:", "")

# Abort flags (matching Kaggle submission verbatim)
os.environ["VESSEL_ABORT_ON_SPARSE_FAIL"]        = "1"
os.environ["VESSEL_ABORT_MIN_ALL_DIMS_MM"]       = "140"
os.environ["VESSEL_ABORT_ON_SMALL_ROI"]          = "1"
os.environ["VESSEL_MIN_ROI_VOXELS"]              = "1000000"
os.environ["VESSEL_ABORT_ON_LOW_UNION"]          = "1"
os.environ["VESSEL_MIN_UNION_SUM"]               = "5000"

# Fallback probabilities on error (14 values, matching Kaggle submission)
os.environ["RSNA_ERROR_FALLBACK_PROBS"]          = "0.02,0.02,0.08,0.08,0.03,0.03,0.07,0.02,0.02,0.02,0.02,0.02,0.02,0.35"

# Model directories
os.environ["VESSEL_NNUNET_SPARSE_MODEL_DIR"]     = VESSEL_SPARSE_MODEL_DIR
os.environ["VESSEL_NNUNET_MODEL_DIR"]            = VESSEL_PRIMARY_MODEL_DIR
os.environ["VESSEL_ADDITIONAL_DENSE_MODEL_DIRS"] = VESSEL_ADDL_MODEL_DIRS
os.environ["VESSEL_FOLDS"]                       = "all"

# Orientation correction (enabled in Kaggle submission)
os.environ["VESSEL_ENABLE_ORIENTATION_CORRECTION"] = "1"
os.environ["VESSEL_ORIENTATION_WEIGHTS"]           = "1,1,1"

# ROI physical extent
os.environ["VESSEL_SPARSE_ROI_EXTENT_MM"]        = "140"
os.environ["VESSEL_REFINE_Z_ONLY"]               = "0"

# ROI margin refinement
os.environ["VESSEL_REFINE_MARGIN_Z"]             = "15"
os.environ["VESSEL_REFINE_MARGIN_XY"]            = "30"

# Sparse / dense overlap
os.environ["VESSEL_SPARSE_OVERLAP"]              = "0.2"
os.environ["VESSEL_DENSE_OVERLAP"]               = "0.3"

# ── ROI classifier ─────────────────────────────────────────────────────────
os.environ["ROI_EXPERIMENTS"]                    = ROI_EXPERIMENT
os.environ["ROI_FOLDS"]                          = ROI_FOLDS
os.environ["ROI_TTA"]                            = ROI_TTA

# ROI nnUNet model directory override (fixes hardcoded /workspace/ paths in configs)
os.environ["ROI_NNUNET_MODEL_DIR"]               = VESSEL_PRIMARY_MODEL_DIR

# Config directory (required by load_experiment_config)
os.environ["CONFIG_DIR"]                         = CONFIG_DIR

# Run mode: "kaggle" enables path overrides for ROI nnUNet backbone
os.environ["RUN_MODE"]                           = "kaggle"

# Additional score switches
os.environ["RSNA_ONLY_AP"]                       = "0"
os.environ["RSNA_ONLY_LOCATIONS"]                = "0"

# Error tracking
os.environ["RSNA_MAX_PREDICT_ERRORS"]            = "300"

# Debug / timing
os.environ["RSNA_DEBUG"]                         = RSNA_DEBUG

# Error log: write to a local file
os.environ["RSNA_ERROR_LOG"]                     = str(REPO_ROOT / "predict_errors.jsonl")

print("Environment variables set.")
print(f"nnUNet_raw        : {os.environ['nnUNet_raw']}")
print(f"nnUNet_preprocessed : {os.environ['nnUNet_preprocessed']}")
print(f"nnUNet_results    : {os.environ['nnUNet_results']}")
print(f"ROI_NNUNET_MODEL_DIR: {os.environ['ROI_NNUNET_MODEL_DIR']}")
print(f"RUN_MODE          : {os.environ['RUN_MODE']}")

## Cell 4 — Import predict function 

In [5]:
from scripts.rsna_submission_roi import predict

print("predict() imported successfully from scripts.rsna_submission_roi")

## Cell 5 — Monkey-patch checkpoint path (identical to Pretrained.ipynb)

In [ ]:
import re
from scripts.rsna_submission_roi import RsnaRoiPipeline

# Save original method
_original_find_ckpt_path = RsnaRoiPipeline._find_ckpt_path

def _find_experiment_dir(experiment_name: str, repo_root: Path):
    """Search for experiment directory in repo root (named 251013-* pattern)"""
    if not repo_root.exists():
        return None
    for candidate_dir in repo_root.glob("251013-*"):
        if not candidate_dir.is_dir():
            continue
        exp_path = candidate_dir / experiment_name
        if exp_path.is_dir():
            return exp_path
    return None

def _patched_find_ckpt_path(self, log_dir: Path):
    """Patched version: tries multiple locations to find checkpoints"""
    mode = (self.opts.roi_ckpt or "last").strip().lower()
    search_dirs = [log_dir]

    if (log_dir.parent / "checkpoint").exists():
        search_dirs.append(log_dir.parent / "checkpoint")

    try:
        if hasattr(self, 'opts') and hasattr(self.opts, 'roi_experiments'):
            exp_name = self.opts.roi_experiments[0] if self.opts.roi_experiments else None
            if exp_name:
                exp_dir = _find_experiment_dir(exp_name, REPO_ROOT)
                if exp_dir:
                    fold_match = re.search(r'fold(\d+)', log_dir.name)
                    if fold_match:
                        fold_num = fold_match.group(1)
                        exp_ckpt_dir = exp_dir / "checkpoint" / f"fold{fold_num}"
                        if exp_ckpt_dir.exists():
                            search_dirs.append(exp_ckpt_dir)
    except Exception:
        pass

    if mode == "best":
        for search_dir in search_dirs:
            epoch_ckpts = list(search_dir.glob("epoch_*.ckpt"))
            if epoch_ckpts:
                def _epoch_num(p: Path) -> int:
                    m = re.search(r"epoch[_=]?(\d+)", p.name)
                    return int(m.group(1)) if m else -1
                return max(epoch_ckpts, key=_epoch_num)
        return None

    # Default: last.ckpt
    for search_dir in search_dirs:
        last_ckpt = search_dir / "last.ckpt"
        if last_ckpt.exists():
            if self.opts.debug:
                import logging
                logging.getLogger(__name__).debug(f"[DEBUG] Found checkpoint: {last_ckpt}")
            return last_ckpt
    return None

# Apply the patch
RsnaRoiPipeline._find_ckpt_path = _patched_find_ckpt_path
print("Applied enhanced checkpoint path patch to search repo root for experiments")

## Cell 6 — Load ground-truth CSV and build test patient list

In [7]:
import pandas as pd
import numpy as np

# Load ground-truth CSV
gt_df = pd.read_csv(TRAIN_CSV_PATH)
print(f"CSV total rows : {len(gt_df)}")
print(f"CSV columns    : {gt_df.columns.tolist()}")

# Fast lookup by SeriesInstanceUID
gt_lookup = gt_df.set_index('SeriesInstanceUID')

# Label columns — must match ANEURYSM_CLASSES order exactly (13 locations + presence)
LOCATION_COLS = [
    'Left Infraclinoid Internal Carotid Artery',
    'Right Infraclinoid Internal Carotid Artery',
    'Left Supraclinoid Internal Carotid Artery',
    'Right Supraclinoid Internal Carotid Artery',
    'Left Middle Cerebral Artery',
    'Right Middle Cerebral Artery',
    'Anterior Communicating Artery',
    'Left Anterior Cerebral Artery',
    'Right Anterior Cerebral Artery',
    'Left Posterior Communicating Artery',
    'Right Posterior Communicating Artery',
    'Basilar Tip',
    'Other Posterior Circulation',
]
PRESENCE_COL   = 'Aneurysm Present'
ALL_LABEL_COLS = LOCATION_COLS + [PRESENCE_COL]   # 14 columns total

# List all DICOM folders in Training_Data and match with CSV
all_folders = sorted(os.listdir(TRAINING_DATA_ROOT))
print(f"\nDICOM folders found in Training_Data : {len(all_folders)}")

valid_patients, missing_in_csv = [], []
for folder in all_folders:
    if folder in gt_lookup.index:
        valid_patients.append(folder)
    else:
        missing_in_csv.append(folder)

if missing_in_csv:
    print(f"WARNING: {len(missing_in_csv)} folders NOT found in CSV (will be skipped)")
    for uid in missing_in_csv:
        print(f"  {uid}")

print(f"Valid patients (folder + CSV match) : {len(valid_patients)}")

# Apply the patient index slice
test_patients = valid_patients[PATIENT_START:PATIENT_END]
print(f"\nTest set after slice [{PATIENT_START}:{PATIENT_END}] : {len(test_patients)} patients")

# Quick label statistics for the test set
test_labels = gt_lookup.loc[test_patients, ALL_LABEL_COLS].values.astype(int)
print(f"  Aneurysm Present = 1 : {test_labels[:, -1].sum()}")
print(f"  Aneurysm Present = 0 : {(1 - test_labels[:, -1]).sum()}")
print(f"  Location positives   : {test_labels[:, :13].sum(axis=0).tolist()}")

## Cell 7 — Pilot run on 5 patients (validate pipeline before full run)

In [8]:
import gc
import torch
from tqdm import tqdm

print('=' * 70)
print('PILOT RUN — Testing pipeline on first 5 patients')
print('=' * 70)

PILOT_N = 5
pilot_ok = True

pbar = tqdm(test_patients[:PILOT_N], desc='Pilot Testing')
for i, series_uid in enumerate(pbar):
    pbar.set_postfix({'patient': series_uid[:15] + '...'})
    dicom_path = str(TRAINING_DATA_ROOT / series_uid)
    true_labels = gt_lookup.loc[series_uid, ALL_LABEL_COLS].values.astype(float)
    
    try:
        # The predict function has its own logging, but we silenced levels globally.
        result_df = predict(dicom_path)
        probs = result_df.to_numpy().flatten()
        del result_df
    except Exception as e:
        tqdm.write(f'Error on {series_uid}: {e}')
        pilot_ok = False
        
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print('\nPilot Complete.' if pilot_ok else 'Pilot failed.')


## Cell 8 — Full inference on all 100 patients [PATIENT_START:PATIENT_END]

> **Memory strategy**: `predict()` is called once per patient. Only the 14 probabilities are retained; the volume is discarded immediately. `gc.collect()` + `torch.cuda.empty_cache()` are called after every patient.

In [ ]:
import time
from tqdm import tqdm

print('=' * 70)
print(f'FULL INFERENCE — {len(test_patients)} patients')
print('=' * 70)

all_results = []
failed_uids = []
global_start = time.time()

pbar = tqdm(test_patients, desc='Total Inference Progress')
for i, series_uid in enumerate(pbar):
    pbar.set_postfix({'patient': series_uid[:15] + '...'})
    dicom_path = str(TRAINING_DATA_ROOT / series_uid)
    true_labels = gt_lookup.loc[series_uid, ALL_LABEL_COLS].values.astype(float)
    
    try:
        result_df = predict(dicom_path)
        probs = result_df.to_numpy().flatten()
        del result_df
        all_results.append({'series_uid': series_uid, 'probs': probs, 'true_labels': true_labels})
    except Exception as e:
        failed_uids.append(series_uid)
        all_results.append({'series_uid': series_uid, 'probs': np.full(14, np.nan), 'true_labels': true_labels})
        
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

total_min = (time.time() - global_start) / 60
print(f'\nInference complete in {total_min:.1f} min')


## Cell 9 — Compute metrics (Accuracy, Precision, Recall, F1, AUC-ROC)

In [24]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

# ── Filter failed predictions ──────────────────────────────────────────────
#valid   = [r for r in all_results if not np.isnan(r['probs']).any()]
valid=all_results
skipped = len(all_results) - len(valid)
N       = len(valid)
print(f"Samples used for metrics : {N}  (skipped {skipped} failed predictions)")

# ── Build arrays ──────────────────────────────────────────────────────────
probs_all = np.stack([r['probs']       for r in valid])   # (N, 14)
true_all  = np.stack([r['true_labels'] for r in valid])   # (N, 14)

loc_probs = probs_all[:, :13]               # (N, 13)
loc_true  = true_all[:,  :13].astype(int)   # (N, 13)
loc_pred  = (loc_probs >= 0.5).astype(int)  # (N, 13)  threshold = 0.5

pres_probs = probs_all[:, 13]              # (N,)
pres_true  = true_all[:,  13].astype(int)  # (N,)
pres_pred  = (pres_probs >= 0.5).astype(int)

THRESHOLD = 0.5

# ── ANEURYSM PRESENT — Binary metrics ─────────────────────────────────────
print("\n" + "=" * 70)
print("  ANEURYSM PRESENT — Binary Classification Metrics")
print("=" * 70)

ap_acc  = accuracy_score(pres_true, pres_pred)
ap_prec = precision_score(pres_true, pres_pred, zero_division=0)
ap_rec  = recall_score(pres_true, pres_pred, zero_division=0)
ap_f1   = f1_score(pres_true, pres_pred, zero_division=0)
ap_auc  = roc_auc_score(pres_true, pres_probs) if len(np.unique(pres_true)) > 1 else float('nan')

print(f"  Patients : {N}  (positive={pres_true.sum()}, negative={N - pres_true.sum()})")
print(f"  Threshold: {THRESHOLD}")
print(f"  Accuracy : {ap_acc:.4f}")
print(f"  Precision: {ap_prec:.4f}")
print(f"  Recall   : {ap_rec:.4f}")
print(f"  F1 Score : {ap_f1:.4f}")
print(f"  AUC-ROC  : {ap_auc:.4f}")

# ── LOCATIONS — Per-location binary metrics ────────────────────────────────
print("\n" + "=" * 70)
print("  LOCATION — Per-Location Binary Metrics  (threshold=0.5)")
print("=" * 70)

loc_metrics_list = []
for j, col in enumerate(LOCATION_COLS):
    yt  = loc_true[:, j]
    yp  = loc_pred[:, j]
    ypr = loc_probs[:, j]

    acc  = accuracy_score(yt, yp)
    prec = precision_score(yt, yp, zero_division=0)
    rec  = recall_score(yt, yp, zero_division=0)
    f1   = f1_score(yt, yp, zero_division=0)
    auc  = roc_auc_score(yt, ypr) if len(np.unique(yt)) > 1 else float('nan')
    loc_metrics_list.append(dict(
        label=col, acc=acc, prec=prec, rec=rec, f1=f1, auc=auc, pos=int(yt.sum())
    ))

hdr = f"{'Location':<47} {'#Pos':>5} {'Acc':>6} {'Prec':>6} {'Rec':>6} {'F1':>6} {'AUC':>6}"
print(hdr)
print("-" * len(hdr))
for m in loc_metrics_list:
    auc_s = f"{m['auc']:.4f}" if not np.isnan(m['auc']) else "  N/A "
    print(f"{m['label']:<47} {m['pos']:>5} {m['acc']:>6.4f} {m['prec']:>6.4f} "
          f"{m['rec']:>6.4f} {m['f1']:>6.4f} {auc_s:>6}")

macro_acc  = np.mean([m['acc']  for m in loc_metrics_list])
macro_prec = np.mean([m['prec'] for m in loc_metrics_list])
macro_rec  = np.mean([m['rec']  for m in loc_metrics_list])
macro_f1   = np.mean([m['f1']   for m in loc_metrics_list])
v_aucs     = [m['auc'] for m in loc_metrics_list if not np.isnan(m['auc'])]
macro_auc  = np.mean(v_aucs) if v_aucs else float('nan')

print("-" * len(hdr))
print(f"{'MACRO AVERAGE':<47} {'':>5} {macro_acc:>6.4f} {macro_prec:>6.4f} "
      f"{macro_rec:>6.4f} {macro_f1:>6.4f} {macro_auc:>6.4f}")

# ── Quick combined summary ─────────────────────────────────────────────────
print("\n" + "=" * 70)
print("  COMBINED SUMMARY")
print("=" * 70)
print(f"  Aneurysm Present  — Acc:{ap_acc:.4f}  Prec:{ap_prec:.4f}  "
      f"Rec:{ap_rec:.4f}  F1:{ap_f1:.4f}  AUC:{ap_auc:.4f}")
print(f"  Location (macro)  — Acc:{macro_acc:.4f}  Prec:{macro_prec:.4f}  "
      f"Rec:{macro_rec:.4f}  F1:{macro_f1:.4f}  AUC:{macro_auc:.4f}")

## Cell 10 — Confusion matrix: Aneurysm Present (2×2)

In [17]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm_pres = confusion_matrix(pres_true, pres_pred, labels=[0, 1])

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_pres, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Predicted\nNegative', 'Predicted\nPositive'],
    yticklabels=['True\nNegative', 'True\nPositive'],
    ax=ax, linewidths=1.5, linecolor='gray',
    annot_kws={"size": 18, "weight": "bold"}
)
ax.set_title(
    f'Aneurysm Present — 2\u00d72 Confusion Matrix\n'
    f'Acc={ap_acc:.3f}  Prec={ap_prec:.3f}  Rec={ap_rec:.3f}  '
    f'F1={ap_f1:.3f}  AUC={ap_auc:.3f}  (N={N})',
    fontsize=11, fontweight='bold', pad=10
)
ax.set_ylabel('Ground Truth', fontsize=12)
ax.set_xlabel(f'Prediction (threshold={THRESHOLD})', fontsize=12)
plt.tight_layout()
save_path_pres = str(REPO_ROOT / 'confusion_matrix_presence_2x2.png')
plt.savefig(save_path_pres, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path_pres}")

## Cell 11 — Confusion matrix: Locations (13×13)

> **Scope**: Only patients where at least one ground-truth location label is 1 (i.e. positive patients).  
> **True class**: `argmax` of the 13 binary ground-truth labels (dominant location if multi-label).  
> **Predicted class**: `argmax` of the 13 predicted probabilities.

In [18]:
from sklearn.metrics import confusion_matrix

# Filter to patients with at least one positive location
pos_mask      = loc_true.sum(axis=1) > 0
loc_true_pos  = loc_true[pos_mask]    # (P, 13)
loc_probs_pos = loc_probs[pos_mask]   # (P, 13)
P             = int(pos_mask.sum())
print(f"Positive-location patients: {P} / {N}")

true_cls = loc_true_pos.argmax(axis=1)   # (P,)
pred_cls = loc_probs_pos.argmax(axis=1)  # (P,)

cm_loc = confusion_matrix(true_cls, pred_cls, labels=list(range(13)))

SHORT_LABELS = [
    'L-IC\nInfra', 'R-IC\nInfra',
    'L-IC\nSupra', 'R-IC\nSupra',
    'L-MCA',       'R-MCA',
    'AComA',
    'L-ACA',       'R-ACA',
    'L-PComA',     'R-PComA',
    'Basilar\nTip','Other\nPost',
]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(
    cm_loc, annot=True, fmt='d', cmap='YlOrRd',
    xticklabels=SHORT_LABELS, yticklabels=SHORT_LABELS,
    ax=ax, linewidths=0.5, linecolor='lightgray',
    annot_kws={"size": 10}
)
ax.set_title(
    f'Location — 13\u00d713 Confusion Matrix  (N={P} positive patients)\n'
    f'Row = true location (argmax of GT labels)   '
    f'Col = predicted location (argmax of probs)',
    fontsize=13, fontweight='bold', pad=12
)
ax.set_ylabel('Ground Truth Location', fontsize=12)
ax.set_xlabel('Predicted Location', fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=9)
plt.tight_layout()
save_path_loc = str(REPO_ROOT / 'confusion_matrix_location_13x13.png')
plt.savefig(save_path_loc, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {save_path_loc}")

## Cell 12 — Final summary

In [19]:
print("=" * 70)
print("  FINAL EVALUATION SUMMARY")
print("=" * 70)
print(f"  Patients tested  : {N}  (from index [{PATIENT_START}:{PATIENT_END}])")
print(f"  Failed / skipped : {skipped}")
print(f"  Threshold used   : {THRESHOLD}")
print()
print(f"  ── Aneurysm Present (binary) ──────────────────────────────────")
print(f"     Accuracy   : {ap_acc:.4f}")
print(f"     Precision  : {ap_prec:.4f}")
print(f"     Recall     : {ap_rec:.4f}")
print(f"     F1 Score   : {ap_f1:.4f}")
print(f"     AUC-ROC    : {ap_auc:.4f}")
print()
print(f"  ── Location (13 classes, macro-average) ────────────────────────")
print(f"     Accuracy   : {macro_acc:.4f}")
print(f"     Precision  : {macro_prec:.4f}")
print(f"     Recall     : {macro_rec:.4f}")
print(f"     F1 Score   : {macro_f1:.4f}")
print(f"     AUC-ROC    : {macro_auc:.4f}")
print()
print("  Saved plots:")
print(f"     {save_path_pres}")
print(f"     {save_path_loc}")
print("=" * 70)